# Lab 2: Logistic Regression, LDA, QDA, and KNN
ISLP Chapter 4 Lab — Smarket, Caravan, and Bikeshare data sets.

## 4.7.1 The Stock Market Data

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                          summarize)

In [ ]:
from ISLP import confusion_table
from ISLP.models import contrast
from sklearn.discriminant_analysis import \
     (LinearDiscriminantAnalysis as LDA,
      QuadraticDiscriminantAnalysis as QDA)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [ ]:
Smarket = load_data('Smarket')
Smarket

In [ ]:
Smarket.columns

In [ ]:
Smarket.corr()

In [ ]:
Smarket.plot(y='Volume');

## 4.7.2 Logistic Regression

In [ ]:
allvars = Smarket.columns.drop(['Today', 'Direction', 'Year'])
design = MS(allvars)
X = design.fit_transform(Smarket)
y = Smarket.Direction == 'Up'
glm = sm.GLM(y,
             X,
             family=sm.families.Binomial())
results = glm.fit()
summarize(results)

In [ ]:
results.params

In [ ]:
results.pvalues

In [ ]:
probs = results.predict()
probs[:10]

In [ ]:
labels = np.array(['Down']*1250)
labels[probs > 0.5] = "Up"

In [ ]:
confusion_table(labels, Smarket.Direction)

In [ ]:
(507+145)/1250, np.mean(labels == Smarket.Direction)

In [ ]:
train = (Smarket.Year < 2005)
Smarket_train = Smarket.loc[train]
Smarket_test = Smarket.loc[~train]
Smarket_test.shape

In [ ]:
X_train, X_test = X.loc[train], X.loc[~train]
y_train, y_test = y.loc[train], y.loc[~train]
glm_train = sm.GLM(y_train,
                    X_train,
                    family=sm.families.Binomial())
results = glm_train.fit()
probs = results.predict(exog=X_test)

In [ ]:
D = Smarket.Direction
L_train, L_test = D.loc[train], D.loc[~train]

In [ ]:
labels = np.array(['Down']*252)
labels[probs > 0.5] = 'Up'
confusion_table(labels, L_test)

In [ ]:
np.mean(labels == L_test), np.mean(labels != L_test)

In [ ]:
model = MS(['Lag1', 'Lag2']).fit(Smarket)
X = model.transform(Smarket)
X_train, X_test = X.loc[train], X.loc[~train]
glm_train = sm.GLM(y_train,
                    X_train,
                    family=sm.families.Binomial())
results = glm_train.fit()
probs = results.predict(exog=X_test)
labels = np.array(['Down']*252)
labels[probs > 0.5] = 'Up'
confusion_table(labels, L_test)

In [ ]:
(35+106)/252, 106/(106+76)

In [ ]:
newdata = pd.DataFrame({'Lag1':[1.2, 1.5],
                         'Lag2':[1.1, -0.8]})
newX = model.transform(newdata)
results.predict(newX)

## 4.7.3 Linear Discriminant Analysis

In [ ]:
lda = LDA(store_covariance=True)

In [ ]:
X_train, X_test = [M.drop(columns=['intercept'])
                    for M in [X_train, X_test]]
lda.fit(X_train, L_train)

In [ ]:
lda.means_

In [ ]:
lda.classes_

In [ ]:
lda.priors_

In [ ]:
lda.scalings_

In [ ]:
lda_pred = lda.predict(X_test)

In [ ]:
confusion_table(lda_pred, L_test)

In [ ]:
lda_prob = lda.predict_proba(X_test)
np.all(
       np.where(lda_prob[:,1] >= 0.5, 'Up','Down') == lda_pred
       )

In [ ]:
np.all(
       [lda.classes_[i] for i in np.argmax(lda_prob, 1)] ==
       lda_pred
       )

In [ ]:
np.sum(lda_prob[:,0] > 0.9)

## 4.7.4 Quadratic Discriminant Analysis

In [ ]:
qda = QDA(store_covariance=True)
qda.fit(X_train, L_train)

In [ ]:
qda.means_, qda.priors_

In [ ]:
qda.covariance_[0]

In [ ]:
qda_pred = qda.predict(X_test)
confusion_table(qda_pred, L_test)

In [ ]:
np.mean(qda_pred == L_test)

## 4.7.5 Naive Bayes

In [ ]:
NB = GaussianNB()
NB.fit(X_train, L_train)

In [ ]:
NB.classes_

In [ ]:
NB.class_prior_

In [ ]:
NB.theta_

In [ ]:
NB.var_

In [ ]:
X_train[L_train == 'Down'].mean()

In [ ]:
X_train[L_train == 'Down'].var(ddof=0)

In [ ]:
nb_labels = NB.predict(X_test)
confusion_table(nb_labels, L_test)

In [ ]:
NB.predict_proba(X_test)[:5]

## 4.7.6 K-Nearest Neighbors

In [ ]:
knn1 = KNeighborsClassifier(n_neighbors=1)
knn1.fit(X_train, L_train)
knn1_pred = knn1.predict(X_test)
confusion_table(knn1_pred, L_test)

In [ ]:
(83+43)/252, np.mean(knn1_pred == L_test)

In [ ]:
knn3 = KNeighborsClassifier(n_neighbors=3)
knn3_pred = knn3.fit(X_train, L_train).predict(X_test)
np.mean(knn3_pred == L_test)

### KNN on the Caravan Data Set

In [ ]:
Caravan = load_data('Caravan')
Purchase = Caravan.Purchase
Purchase.value_counts()

In [ ]:
348 / 5822

In [ ]:
feature_df = Caravan.drop(columns=['Purchase'])

In [ ]:
scaler = StandardScaler(with_mean=True,
                         with_std=True,
                         copy=True)

In [ ]:
scaler.fit(feature_df)
X_std = scaler.transform(feature_df)

In [ ]:
feature_std = pd.DataFrame(
                  X_std,
                  columns=feature_df.columns);
feature_std.std()

In [ ]:
(X_train,
 X_test,
 y_train,
 y_test) = train_test_split(feature_std,
                             Purchase,
                             test_size=1000,
                             random_state=0)

In [ ]:
knn1 = KNeighborsClassifier(n_neighbors=1)
knn1_pred = knn1.fit(X_train, y_train).predict(X_test)
np.mean(y_test != knn1_pred), np.mean(y_test != "No")

In [ ]:
confusion_table(knn1_pred, y_test)

In [ ]:
9/(53+9)

### Tuning Parameters

In [ ]:
for K in range(1,6):
    knn = KNeighborsClassifier(n_neighbors=K)
    knn_pred = knn.fit(X_train, y_train).predict(X_test)
    C = confusion_table(knn_pred, y_test)
    templ = ('K={0:d}: # predicted to rent: {1:>2},' +
             ' # who did rent {2:d}, accuracy {3:.1%}')
    pred = C.loc['Yes'].sum()
    did_rent = C.loc['Yes','Yes']
    print(templ.format(
          K,
          pred,
          did_rent,
          did_rent / pred))

### Comparison to Logistic Regression

In [ ]:
logit = LogisticRegression(C=1e10, solver='liblinear')
logit.fit(X_train, y_train)
logit_pred = logit.predict_proba(X_test)
logit_labels = np.where(logit_pred[:,1] > 0.5, 'Yes', 'No')
confusion_table(logit_labels, y_test)

In [ ]:
logit_labels = np.where(logit_pred[:,1] > 0.25, 'Yes', 'No')
confusion_table(logit_labels, y_test)

In [ ]:
9/(20+9)

## 4.7.7 Linear and Poisson Regression on the Bikeshare Data

In [ ]:
Bike = load_data('Bikeshare')

In [ ]:
Bike.shape, Bike.columns

### Linear Regression

In [ ]:
X = MS(['mnth',
        'hr',
        'workingday',
        'temp',
        'weathersit']).fit_transform(Bike)
Y = Bike['bikers']
M_lm = sm.OLS(Y, X).fit()
summarize(M_lm)

In [ ]:
hr_encode = contrast('hr', 'sum')
mnth_encode = contrast('mnth', 'sum')

In [ ]:
X2 = MS([mnth_encode,
         hr_encode,
         'workingday',
         'temp',
         'weathersit']).fit_transform(Bike)
M2_lm = sm.OLS(Y, X2).fit()
S2 = summarize(M2_lm)
S2

In [ ]:
np.sum((M_lm.fittedvalues - M2_lm.fittedvalues)**2)

In [ ]:
np.allclose(M_lm.fittedvalues, M2_lm.fittedvalues)

In [ ]:
coef_month = S2[S2.index.str.contains('mnth')]['coef']
coef_month

In [ ]:
months = Bike['mnth'].dtype.categories
coef_month = pd.concat([
                coef_month,
                pd.Series([-coef_month.sum()],
                          index=['mnth[Dec]'])
                ])
coef_month

In [ ]:
fig_month, ax_month = subplots(figsize=(8,8))
x_month = np.arange(coef_month.shape[0])
ax_month.plot(x_month, coef_month, marker='o', ms=10)
ax_month.set_xticks(x_month)
ax_month.set_xticklabels([l[5] for l in coef_month.index], fontsize=20)
ax_month.set_xlabel('Month', fontsize=20)
ax_month.set_ylabel('Coefficient', fontsize=20);

In [ ]:
coef_hr = S2[S2.index.str.contains('hr')]['coef']
coef_hr = coef_hr.reindex(['hr[{0}]'.format(h) for h in range(23)])
coef_hr = pd.concat([coef_hr,
                      pd.Series([-coef_hr.sum()], index=['hr[23]'])
                      ])

In [ ]:
fig_hr, ax_hr = subplots(figsize=(8,8))
x_hr = np.arange(coef_hr.shape[0])
ax_hr.plot(x_hr, coef_hr, marker='o', ms=10)
ax_hr.set_xticks(x_hr[::2])
ax_hr.set_xticklabels(range(24)[::2], fontsize=20)
ax_hr.set_xlabel('Hour', fontsize=20)
ax_hr.set_ylabel('Coefficient', fontsize=20);

### Poisson Regression

In [ ]:
M_pois = sm.GLM(Y, X2, family=sm.families.Poisson()).fit()

In [ ]:
S_pois = summarize(M_pois)
coef_month = S_pois[S_pois.index.str.contains('mnth')]['coef']
coef_month = pd.concat([coef_month,
                         pd.Series([-coef_month.sum()],
                                   index=['mnth[Dec]'])])
coef_hr = S_pois[S_pois.index.str.contains('hr')]['coef']
coef_hr = pd.concat([coef_hr,
                      pd.Series([-coef_hr.sum()],
                                index=['hr[23]'])])

In [ ]:
fig_pois, (ax_month, ax_hr) = subplots(1, 2, figsize=(16,8))
ax_month.plot(x_month, coef_month, marker='o', ms=10)
ax_month.set_xticks(x_month)
ax_month.set_xticklabels([l[5] for l in coef_month.index], fontsize=20)
ax_month.set_xlabel('Month', fontsize=20)
ax_month.set_ylabel('Coefficient', fontsize=20)
ax_hr.plot(x_hr, coef_hr, marker='o', ms=10)
ax_hr.set_xticklabels(range(24)[::2], fontsize=20)
ax_hr.set_xlabel('Hour', fontsize=20)
ax_hr.set_ylabel('Coefficient', fontsize=20);

In [ ]:
fig, ax = subplots(figsize=(8, 8))
ax.scatter(M2_lm.fittedvalues,
           M_pois.fittedvalues,
           s=20)
ax.set_xlabel('Linear Regression Fit', fontsize=20)
ax.set_ylabel('Poisson Regression Fit', fontsize=20)
ax.axline([0,0], c='black', linewidth=3,
          linestyle='--', slope=1);